In [26]:
import numpy as np
import pandas as pd
from IPython.display import FileLink
from pytplot import get_data
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from adjustText import adjust_text

%matplotlib tk

In [15]:
file_path = 'merged.csv'
merged = pd.read_csv(file_path)
merged

,Start time_stereo,End time_stereo,High frequency (Hz)_stereo,Low frequency (Hz)_stereo,Notes_stereo,Frequency Change_stereo,Duration (seconds)_stereo,Drift Rate (Hz/s)_stereo,Date,Source_stereo,...,Source_psp,Start Time (UTC),End Time (UTC),Start Energy (keV),End Energy (keV),Instrument,Notes,Duration (seconds),Energy Change (keV),Source
0,2021-09-21 09:43:00,2021-09-21 10:50:00,15000000,40000,NaN,14960000,4020.0,3721.393035,2021-09-21,STEREO,...,PSP,2021-09-21 10:50:00,2021-09-21 12:05:00,80,20,SEP,NaN,4500.0,60,MAVEN
1,2021-10-28 15:30:00,2021-10-28 19:15:00,15000000,20000,NaN,14980000,13500.0,1109.629630,2021-10-28,STEREO,...,PSP,2021-10-28 16:55:00,2021-10-28 18:37:00,100,20,SEP,leads into a hefty SEP electron event so mayb...,6120.0,80,MAVEN
2,2022-02-15 21:55:00,2022-02-16 00:25:00,10000000,40000,NaN,9960000,9000.0,1106.666667,2022-02-15,STEREO,...,PSP,2022-02-15 22:55:00,2022-02-16 01:00:00,100,20,SEP,leads into a SEP event so maybe flare but poo...,7500.0,80,MAVEN
3,2022-05-08 02:00:00,2022-05-08 02:50:00,5000000,100000,NaN,4900000,3000.0,1633.333333,2022-05-08,STEREO,...,PSP,2022-05-08 05:25:00,2022-05-08 06:30:00,80,20,SEP,NaN,3900.0,60,MAVEN
4,2022-05-17 20:37:00,2022-05-17 21:54:00,2300000,70000,NaN,2230000,4620.0,482.683983,2022-05-17,STEREO,...,PSP,2022-05-17 22:15:00,2022-05-18 00:30:00,60,20,SEP,NaN,8100.0,40,MAVEN
5,2022-06-29 17:37:00,2022-06-29 18:00:00,1000000,70000,NaN,930000,1380.0,673.913043,2022-06-29,STEREO,...,PSP,2022-06-29 20:45:00,2022-06-29 21:30:00,80,20,SEP,only seen in 2F,2700.0,60,MAVEN
6,2022-06-30 05:05:00,2022-06-30 05:37:00,6700000,70000,NaN,6630000,1920.0,3453.125000,2022-06-30,STEREO,...,PSP,2022-06-30 05:30:00,2022-06-30 08:30:00,100,20,SEP,on top of minor SEP event,10800.0,80,MAVEN
7,2022-07-01 11:37:00,2022-07-01 12:08:00,15000000,100000,NaN,14900000,1860.0,8010.752688,2022-07-01,STEREO,...,PSP,2022-07-01 16:25:00,2022-07-01 17:00:00,80,20,SEP,low intensity and on top of minor SEP event,2100.0,60,MAVEN
8,2022-07-22 13:05:00,2022-07-22 13:32:00,3500000,40000,NaN,3460000,1620.0,2135.802469,2022-07-22,STEREO,...,PSP,2022-07-22 17:55:00,2022-07-22 21:05:00,80,20,SEP,very clear signature,11400.0,60,MAVEN
9,2022-11-17 09:16:00,2022-11-17 11:24:00,10000000,50000,NaN,9950000,7680.0,1295.572917,2022-11-17,STEREO,...,PSP,2022-11-17 10:20:00,2022-11-17 12:50:00,40,20,SEP,NaN,9000.0,20,MAVEN


In [95]:
def plot_freq_change_vs_energy_change(df, label1, label2):
    plt.figure(figsize=(10, 6))
    plt.scatter(df["Energy Change (keV)"], df["Frequency Change_stereo"], alpha=0.7, label=label1, color="C0")
    plt.scatter(df["Energy Change (keV)"], df["Frequency Change_psp"], alpha=0.7, label=label2, color="C3")

    texts = []
    for i, txt in enumerate(df["Date"]):
        texts.append(plt.text(df["Energy Change (keV)"].iloc[i], df["Frequency Change_stereo"].iloc[i], str(txt),
                              fontsize=8, alpha=0.75, color="C0"))
    for i, txt in enumerate(df["Date"]):
        texts.append(plt.text(df["Energy Change (keV)"].iloc[i], df["Frequency Change_psp"].iloc[i], str(txt),
                              fontsize=8, alpha=0.75, color="C3"))
    
    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Energy Change (keV) (log scale)")
    plt.ylabel("Frequench Change (Hz) (log scale)")
    # plt.ylim(1e4, 1e9)
    plt.title(f"Log–Log Scatter Plot of Freq Change ({label1} & {label2}) vs Energy Change (Maven)")
    plt.legend()
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [99]:
plot_freq_change_vs_energy_change(merged, "STEREO", "PSP")

In [19]:
def plot_freq_change_vs_energy_change_psp(df, label):
    plt.figure(figsize=(10, 6))
    plt.scatter(df["Energy Change (keV)"], df["Frequency Change_psp"], alpha=0.7, label=label, color="C0")

    texts = []
    for i, txt in enumerate(df["Date"]):
        texts.append(plt.text(df["Energy Change (keV)"].iloc[i], df["Frequency Change_psp"].iloc[i], str(txt),
                              fontsize=8, alpha=0.75, color="C0"))
    
    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Energy Change (keV) (log scale)")
    plt.ylabel("Frequench Change (Hz) (log scale)")
    plt.ylim(1e4, 1e9)
    plt.title(f"Log–Log Scatter Plot of Freq Change ({label}) vs Energy Change (Maven)")
    plt.legend()
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [103]:
plot_freq_change_vs_energy_change_psp(merged, "PSP")

In [21]:
def plot_freq_change_vs_energy_change_stereo(df, label):
    plt.figure(figsize=(10, 6))
    plt.scatter(df["Energy Change (keV)"], df["Frequency Change_stereo"], alpha=0.7, label=label, color="C0")

    texts = []
    for i, txt in enumerate(df["Date"]):
        texts.append(plt.text(df["Energy Change (keV)"].iloc[i], df["Frequency Change_stereo"].iloc[i], str(txt),
                              fontsize=8, alpha=0.75, color="C0"))
    
    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Energy Change (keV) (log scale)")
    plt.ylabel("Frequench Change (Hz) (log scale)")
    plt.ylim(1e4, 1e9)
    plt.title(f"Log–Log Scatter Plot of Freq Change ({label}) vs Energy Change (Maven)")
    plt.legend()
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [93]:
plot_freq_change_vs_energy_change_stereo(merged, "STEREO")

In [41]:
def plot_cubic_fit(plt, df, label, num, text=True):
    color_array = ['C0', 'C3', 'C2', 'C1']
    
    x = df["Energy Change (keV)"].values.reshape(-1, 1)
    y = df[f"Frequency Change_{label}"].values

    poly = PolynomialFeatures(degree=3, include_bias=False)
    x_poly = poly.fit_transform(x)

    # Fit the cubic model
    model = LinearRegression()
    model.fit(x_poly, y)

    # after: model.fit(x_poly, y)
    y_pred = model.predict(x_poly)
    resid  = y - y_pred
    rmsd   = np.sqrt(np.mean(resid**2))
    print(f"[{label}] RMSD = {rmsd:.3g}  (same units as y)")


    x_fit = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
    x_fit_poly = poly.transform(x_fit)
    y_fit = model.predict(x_fit_poly)

    plt.scatter(x, y, alpha=0.7, label=f"Maven vs {label}", color=color_array[num])
    plt.plot(x_fit, y_fit, color=color_array[num + 2], linestyle='--', label=f'Cubic Fit ({label})')

    coef = model.coef_
    intercept = model.intercept_
    print(f"Cubic model for {label}: y = {intercept:.2e} + {coef[0]:.2e}*x + {coef[1]:.2e}*x² + {coef[2]:.2e}*x³")

    if text:
        texts = []
        for i, txt in enumerate(df["Date"]):
            texts.append(plt.text(df["Energy Change (keV)"].iloc[i], df[f"Frequency Change_{label}"].iloc[i], str(txt),
                                  fontsize=8, alpha=0.75, color=color_array[num]))

    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Energy Change (keV)")
    plt.ylabel("Frequency Change (Hz)")
    plt.title("Cubic Fit: Frequency Change vs Energy Change")
    plt.ylim(1e4, 1e9)
    plt.legend()
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.tight_layout()

In [113]:
plt.figure(figsize=(10, 6))
plot_cubic_fit(plt, merged, "stereo", 0)

[stereo] RMSD = 5.11e+06  (same units as y)
Cubic model for stereo: y = 9.62e+06 + 1.22e+05*x + -2.65e+03*x² + 1.17e+01*x³


In [115]:
plt.figure(figsize=(10, 6))
plot_cubic_fit(plt, merged, "psp", 0)

[psp] RMSD = 6.36e+06  (same units as y)
Cubic model for psp: y = 8.47e+06 + 3.63e+05*x + -5.58e+03*x² + 2.18e+01*x³


In [155]:
plt.figure(figsize=(10, 6))
plot_cubic_fit(plt, merged, "stereo", 0, text=False)
plot_cubic_fit(plt, merged, "psp", 1, text=False)

Cubic model for stereo: y = 9.62e+06 + 1.22e+05*x + -2.65e+03*x² + 1.17e+01*x³
Cubic model for psp: y = 8.47e+06 + 3.63e+05*x + -5.58e+03*x² + 2.18e+01*x³


In [127]:
def plot_power_fit(plt, df, label, num, text=True):
    color_array = ['C0', 'C3', 'C2', 'C1']
    plt.scatter(df["Energy Change (keV)"], df[f"Frequency Change_{label}"], alpha=0.7, label=f"Maven vs {label}", color=color_array[num])

    x = df["Energy Change (keV)"].values
    y = df[f"Frequency Change_{label}"].values
    
    log_x = np.log10(x).reshape(-1, 1)
    log_y = np.log10(y)
    model = LinearRegression()
    model.fit(log_x, log_y)
    
    # Generate x range and corresponding fitted y
    x_fit = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
    y_fit = 10 ** (model.intercept_ + model.coef_[0] * np.log10(x_fit))

    log_y_pred = model.predict(log_x)
    y_pred = 10 ** log_y_pred

    resid = y - y_pred
    rmsd_linear = np.sqrt(np.mean(resid**2))
    
    plt.plot(x_fit, y_fit, color=color_array[num+2], linestyle='--', label=f'Best Fit ({label})')
    
    slope = model.coef_[0]
    intercept = model.intercept_

    # .3e is scientific notation with 3 decimal places
    # .3f is 3 digits after the decimal point
    print(f"Power law model for {label}: y = {10**intercept:.3e} * x^{slope:.3f}")

    if text:
        texts = []
        for i, txt in enumerate(df["Date"]):
            texts.append(plt.text(df["Energy Change (keV)"].iloc[i], df[f"Frequency Change_{label}"].iloc[i], str(txt),
                                  fontsize=8, alpha=0.75, color=color_array[num]))

    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Energy Change (keV) (log scale)")
    plt.ylabel("Frequench Change (Hz) (log scale)")
    plt.title("Power Fit: Frequency Change vs Energy Change")
    plt.ylim(1e4, 1e9)
    plt.legend()
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.tight_layout()

In [129]:
plt.figure(figsize=(10, 6))
plot_power_fit(plt, merged, "stereo", 0)

Power law model for stereo: y = 3.455e+07 * x^-0.383


In [107]:
plt.figure(figsize=(10, 6))
plot_power_fit(plt, merged, "psp", 0)

[psp] RMSD = 7.17e+06  (same units as y)
Power law model for psp: y = 2.188e+07 * x^-0.159


In [89]:
plt.figure(figsize=(10, 6))
plot_power_fit(plt, merged, "stereo", 0, text=False)
plot_power_fit(plt, merged, "psp", 1, text=False)

[stereo] RMSD = 5.83e+06  (same units as y)
Power law model for stereo: y = 3.455e+07 * x^-0.383
[psp] RMSD = 7.17e+06  (same units as y)
Power law model for psp: y = 2.188e+07 * x^-0.159
